# `oncoj` Package — Usage Notebook

This notebook demonstrates the five modules in `src/oncoj/`:
- `oncoj.kana` — phonemic-to-katakana conversion
- `oncoj.lemma_id` — lemma ID parsing, comparison, and generation
- `oncoj.dictionary` — dictionary CRUD and querying
- `oncoj.corpus` — corpus line/document parsing and annotation
- `oncoj.tags` — linguistic tag-set reference

All examples run against the real corpus data in `data/`.


## 0. Setup

In [ ]:
import sys
import os

# Allow importing oncoj without installing the package
sys.path.insert(0, os.path.join('..', 'src'))

# Paths to the real data files
DICT_FILE   = os.path.join('..', 'data', 'xml', 'dict', 'dictionary.xml')
CORPUS_FILE = os.path.join('..', 'data', 'xml', 'text', 'EN_01.xml')

print("Python:", sys.version.split()[0])
print("Working directory:", os.getcwd())


---
## 1. `oncoj.kana` — Phonemic-to-Katakana Conversion

Old Japanese is romanised in the corpus using the Frellesvig-Whitman convention.
`phonemic_to_kana()` converts these romanised forms to historical katakana using a
longest-match left-to-right table.


In [2]:
from oncoj.core.kana import phonemic_to_kana

examples = [
    "kamu",       # god
    "nusi",       # lord/master
    "para",       # field/plain
    "motite",     # having / by means of  (ti → チ)
    "takama",     # high heaven
    "wope",       # call out
    "kwo",        # child  (kwo → コ, a 3-char match)
    "mwo",        # also/too
]

for form in examples:
    print(f"{form:12s} → {phonemic_to_kana(form)}")


kamu         → カム
nusi         → ヌシ
para         → ハラ
motite       → モチテ
takama       → タカマ
wope         → ヲヘ
kwo          → コ
mwo          → モ


In [3]:
# Unrecognised characters are flagged with ⟨…⟩
print(repr(phonemic_to_kana("k_a")))   # _ is not a valid phoneme


'⟨k⟩⟨_⟩ア'


---
## 2. `oncoj.lemma_id` — Lemma IDs

Every dictionary entry and annotated corpus line carries a `LemmaID`.  The format is:
`<PREFIX><zero-padded-6-digit-number>[optional-letter-suffix]` — e.g. `L000006a`, `N000001`.


In [4]:
from oncoj.core.lemma_id import LemmaID, IDGenerator

# Parsing
lid = LemmaID.parse("L000006a")
print("prefix :", lid.prefix)
print("number :", lid.number)
print("suffix :", lid.suffix)
print("str    :", lid)


prefix : L
number : 6
suffix : a
str    : L000006a


In [5]:
# Equality — LemmaID compares equal to the equivalent string
print(lid == "L000006a")          # True
print(lid == LemmaID.parse("L000006a"))  # True
print(lid == "L000007")           # False

# Ordering
ids = [LemmaID.parse(s) for s in ["L000010", "L000006a", "L000006", "L000001"]]
print(sorted(ids))


True
True
False
[LemmaID('L000001'), LemmaID('L000006'), LemmaID('L000006a'), LemmaID('L000010')]


In [6]:
# Validity check
print(LemmaID.is_valid("L000006a"))   # True
print(LemmaID.is_valid("N999999"))    # True
print(LemmaID.is_valid("L6"))         # False — not zero-padded
print(LemmaID.is_valid("word"))       # False


True
True
True
False


In [7]:
# LemmaID is immutable
try:
    lid.prefix = "N"
except AttributeError as e:
    print("Immutable:", e)


Immutable: LemmaID is immutable


In [8]:
# IDGenerator: allocates new IDs, skipping already-used numbers
existing = {LemmaID.parse(s) for s in ["L000001", "L000003"]}
gen = IDGenerator(existing=existing, prefix="L")

print("next:", gen.next_id())    # L000002
print("next:", gen.next_id())    # L000004 (skips L000003)
print("next:", gen.next_id())    # L000005
print("peek:", gen.peek_next_number())  # 6 (doesn't advance counter)


next: L000002
next: L000004
next: L000005
peek: 6


In [9]:
# Change prefix on the fly
gen2 = IDGenerator(start=1, prefix="N")
print(gen2.next_id())          # N000001
print(gen2.next_id(prefix="T"))  # T000002  (prefix override for this call)


N000001
T000002


---
## 3. `oncoj.dictionary` — Dictionary

The dictionary contains ~7 000 lemma entries, each with a set of structured fields.
Required fields (in canonical order): `.GLOSS`, `.MEANING`, `.FORM`, `.KANA`, `.POS`.


In [10]:
from oncoj.core.dictionary import Dictionary, DictEntry

# Load the full dictionary from disk
d = Dictionary.from_file(DICT_FILE)
print(f"{len(d)} entries loaded")


7006 entries loaded


In [11]:
# Look up a known entry
e = d["L000006a"]
print("GLOSS   :", e.get_first(".GLOSS"))
print("POS     :", e.get_first(".POS"))
print("FORMs   :", e.get_all(".FORM"))    # multi-valued
print("KANAs   :", e.get_all(".KANA"))    # multi-valued
print("MEANINGs:", e.get_all(".MEANING"))


GLOSS   : NEG
POS     : auxiliary
FORMs   : ['nu', 'zu']
KANAs   : ['ヌ', 'ズ']
MEANINGs: ['[negative]']


In [12]:
# Query by form (handles multiple entries sharing the same form)
hits = d.find_by_form("ki")
print(f"Found {len(hits)} entry/entries for 'ki':")
for h in hits[:5]:
    print(f"  {h.eid}  {h.get_first('.GLOSS')}  ({h.get_first('.POS')})")


Found 5 entry/entries for 'ki':
  L000015  SPST  (auxiliary)
  L030608a  wear  (verb)
  L030612a  come  (verb)
  L050113  fangs; teeth  (noun)
  L052476  alcohol  (noun)


In [14]:
# Query by POS
verbs = d.find_by_pos("verb")
print(f"Entries with POS='verb': {len(verbs)}")

# Substring search
aux_like = d.find_by_pos("aux", exact=False)
print(f"Entries with POS containing 'aux': {len(aux_like)}")


Entries with POS='verb': 1434
Entries with POS containing 'aux': 17


In [16]:
# Query by gloss substring
neg_entries = d.find_by_gloss("NEG", exact=True)
for e in neg_entries[:3]:
    print(e.eid, e.get_first(".GLOSS"), e.get_all(".FORM"))


L000006a NEG ['nu', 'zu']
L000006b NEG ['nana', 'nani']
L000012 NEG ['zari']


In [18]:
# Creating a new entry and adding it
new_entry = DictEntry.blank("L999001", "testword")
new_entry.set(".GLOSS", "TEST")
new_entry.set(".MEANING", "[test meaning]")
new_entry.set(".POS", "noun")
print(new_entry.to_text())


---------------------------------------------------
=== L999001
.GLOSS	TEST
.MEANING	[test meaning]
.FORM	testword
.KANA	テ⟨s⟩ト⟨r⟩⟨d⟩
.POS	noun




In [19]:
# Round-trip: serialise to text and parse back
text = d.to_text()
d2 = Dictionary.from_text(text)
print("Original entries:", len(d))
print("After round-trip:", len(d2))
assert d["L000006a"].get_all(".FORM") == d2["L000006a"].get_all(".FORM")
print("Forms match:", d["L000006a"].get_all(".FORM"))


Original entries: 7006
After round-trip: 7006
Forms match: ['nu', 'zu']


---
## 4. `oncoj.corpus` — Corpus Documents

Each corpus file is a sequence of **utterances** (sentences), each containing
**tree-path lines** (one per syntactic node) and **comment lines** (headers, ID lines,
original-script glosses).


In [21]:
from oncoj.core.corpus import CorpusLine, CorpusDocument
from oncoj.core.lemma_id import LemmaID

# Load a corpus document
doc = CorpusDocument.from_file(CORPUS_FILE)
print(f"Document: {doc.filename}")
print(f"Utterances : {len(doc)}")
print(f"Corpus lines: {len(doc.all_corpus_lines())}")


Document: EN_01.txt
Utterances : 15
Corpus lines: 1219


In [22]:
# Inspect the first utterance
utt = doc[0]
print("Sentence ID:", utt.sentence_id)
print("Header     :", utt.header)
print("Corpus lines:", len(utt.corpus_lines()))
print("Unannotated:", len(utt.unannotated_lines()))
print("Annotated  :", len(utt.annotated_lines()))


Sentence ID: 1_EN_01
Header     : =N(" ugonapar eru kamu nusi papuri ra moromoro kikosi myese to noru ")
Corpus lines: 11
Unannotated: 10
Annotated  : 1


In [23]:
# Look at individual corpus lines
for cl in utt.corpus_lines()[:5]:
    print(f"  form={cl.word_form!r:12s}  phon_tag={cl.phon_tag!r:10s}  lemma={str(cl.lemma_id)!r:12s}  annotated={cl.is_annotated}")


  form='ugonapar'    phon_tag='LOG'       lemma='None'        annotated=False
  form='eru'         phon_tag='LOG'       lemma='None'        annotated=False
  form='kamu'        phon_tag='LOG'       lemma='None'        annotated=False
  form='nusi'        phon_tag='LOG'       lemma='None'        annotated=False
  form='papuri'      phon_tag='LOG'       lemma='None'        annotated=False


In [24]:
# Accessing the syntactic path
cl = utt.corpus_lines()[0]
print("Full fields :", cl.fields)
print("Synt path   :", cl.synt_path)
print("Preceding tag:", cl.preceding_synt_tag())


Full fields : ['IP-MAT', 'IP-ARG', 'IP-REL', 'VB', 'VB-STM', 'LOG', 'ugonapar']
Synt path   : ['IP-MAT', 'IP-ARG', 'IP-REL', 'VB', 'VB-STM']
Preceding tag: VB-STM


In [25]:
# Find lines by word form across the whole document
hits = doc.find_by_form("para")
print(f"'para' appears {len(hits)} time(s):")
for utt_hit, cl in hits:
    print(f"  sentence {utt_hit.sentence_id}: {cl}")


'para' appears 8 time(s):
  sentence 2_EN_01: IP-MAT,IP-NMZ-FRM,PP-DAT,NP,PP-GEN,NP,IP-REL,IP-ADV,NP,IP-REL,PP-DAT,NP,N,LOG,para
  sentence 3_EN_01: IP-MAT,IP-ADV,NP,N,LOG,para
  sentence 4_EN_01: IP-MAT,IP-ADV,PP-TOP,NP,IP-REL,PP-DAT,NP,N,LOG,para
  sentence 4_EN_01: IP-MAT,IP-ADV,PP-TOP,NP,IP-REL,PP-DAT,NP,N,LOG,para
  sentence 7_EN_01: IP-MAT,IP-ARG,IP-ADV,NP,PP-DAT,NP,N,LOG,para
  sentence 10_EN_01: IP-MAT,PP-TOP,NP,N,LOG,para
  sentence 10_EN_01: IP-MAT,IP-ADV,PP-DAT,NP,N,LOG,para
  sentence 14_EN_01: IP-MAT,IP-ARG,IP-ARG,IP-ADV,NP,N,LOG,para


In [26]:
# Find all lines annotated with a specific lemma ID
hits = doc.find_by_lemma("L000520")
print(f"L000520 appears {len(hits)} time(s):")
for utt_hit, cl in hits:
    print(f"  sentence {utt_hit.sentence_id}: {cl}")


L000520 appears 87 time(s):
  sentence 2_EN_01: IP-MAT,IP-NMZ-FRM,PP-DAT,NP,PP-GEN,NP,IP-REL,IP-ADV,NP,IP-REL,PP-DAT,NP,PP-GEN,P-CASE-GEN,L000520,LOG,no
  sentence 2_EN_01: IP-MAT,IP-NMZ-FRM,PP-DAT,NP,PP-GEN,NP,IP-REL,IP-ADV,NP,N,C-N,PP-GEN,P-CASE-GEN,L000520,LOG,no
  sentence 2_EN_01: IP-MAT,IP-NMZ-FRM,PP-DAT,NP,PP-GEN,NP,IP-REL,IP-ADV,NP,N,C-N;@2,PP-GEN,P-CASE-GEN,L000520,LOG,no
  sentence 2_EN_01: IP-MAT,IP-NMZ-FRM,PP-DAT,NP,PP-GEN,NP,P-CASE-GEN,L000520,PHON,no
  sentence 2_EN_01: IP-MAT,IP-ARG,PP-OB1,NP,PP-GEN,P-CASE-GEN,L000520,LOG,no
  sentence 3_EN_01: IP-MAT,IP-NMZ-FRM,PP-DAT,NP,PP-GEN,NP,PP-GEN,P-CASE-GEN,L000520,LOG,no
  sentence 3_EN_01: IP-MAT,IP-NMZ-FRM,PP-DAT,NP,PP-GEN,P-CASE-GEN,L000520,PHON,no
  sentence 3_EN_01: IP-MAT,IP-ADV,PP-OB1,C-PP,IP-REL,PP-GEN,P-CASE-GEN,L000520,PHON,no
  sentence 3_EN_01: IP-MAT,IP-ADV,PP-SBJ,P-CASE-GEN,L000520,PHON,no
  sentence 3_EN_01: IP-MAT,IP-ADV,NP,PP-GEN,P-CASE-GEN,L000520,PHON,no
  sentence 4_EN_01: IP-MAT,IP-ADV,PP-TOP,NP,IP-REL,PP-D

In [27]:
# All distinct lemma IDs in the document
all_ids = doc.all_lemma_ids()
print(f"Distinct lemma IDs: {len(all_ids)}")

# Unannotated forms (word forms still lacking a lemma ID)
ua = doc.unannotated_forms()
if ua:
    print(f"Unannotated forms: {len(ua)}")
    for form, lines in list(ua.items())[:3]:
        print(f"  {form!r}: {len(lines)} line(s)")
else:
    print("All forms are annotated in this file.")


Distinct lemma IDs: 25
Unannotated forms: 286
  'ugonapar': 1 line(s)
  'eru': 3 line(s)
  'kamu': 5 line(s)


In [28]:
# Mutation demo: insert / replace / remove a lemma
sample_line = CorpusLine.parse("IP-MAT,NP,N,LOG,kamu")
print("Before:", sample_line)
sample_line.insert_lemma("L000999")
print("After insert:", sample_line)
sample_line.replace_lemma("L000001")
print("After replace:", sample_line)
old = sample_line.remove_lemma()
print("After remove:", sample_line, "  (removed:", old, ")")


Before: IP-MAT,NP,N,LOG,kamu
After insert: IP-MAT,NP,N,L000999,LOG,kamu
After replace: IP-MAT,NP,N,L000001,LOG,kamu
After remove: IP-MAT,NP,N,LOG,kamu   (removed: L000001 )


In [ ]:
# Round-trip: serialise back to text
doc2 = CorpusDocument.from_text(doc.to_text())
print("Utterance count preserved:", len(doc2) == len(doc))
print("Corpus line count preserved:",
      len(doc2.all_corpus_lines()) == len(doc.all_corpus_lines()))


---
## 5. `oncoj.tags` — Linguistic Tag Reference

`tags.py` centralises all domain constants: writing-mode tags, POS values, inflection
types, text collection codes, and more.


In [30]:
from oncoj.core.tags import (
    PHON_TAGS, MULTI_VALUE_FIELDS, REQUIRED_FIELDS,
    strip_disambig, is_phon_tag,
    DICT_FIELDS, POS_VALUES, ITYPE_VALUES, VAX_SEMANTICS,
    GEO_VALUES, TEXT_COLLECTIONS,
)

print("Writing-mode (PHON) tags:")
for tag in sorted(PHON_TAGS):
    print(f"  {tag}")


Writing-mode (PHON) tags:
  BPHON
  ILL
  LOG
  NLOG
  NLPOG
  ORDLOG
  PHON
  PHON-KUN
  PHON-ON
  PLOG


In [31]:
print("Required dictionary fields (canonical order):")
for f in REQUIRED_FIELDS:
    print(f"  {f:16s}  {DICT_FIELDS[f]}")


Required dictionary fields (canonical order):
  .GLOSS            abbreviated gloss / functional label (e.g. NEG, PRF, CJR)
  .MEANING          full English meaning/gloss (multi-valued for polysemous entries)
  .FORM             romanised phonemic form(s) (multi-valued for irregular paradigms)
  .KANA             historical katakana spelling (parallel to .FORM, auto-generated)
  .POS              part of speech (free-text, e.g. 'noun', 'auxiliary', 'makura kotoba')


In [32]:
print("Multi-valued dictionary fields:")
for f in sorted(MULTI_VALUE_FIELDS):
    print(f"  {f}")


Multi-valued dictionary fields:
  .COMPOUND
  .DERIVATION
  .FORM
  .GEO
  .ITYPE
  .KANA
  .MEANING
  .NOTE
  .POS
  .PTR
  .RELATED
  .TRANSREL
  .VCLASS


In [33]:
# Disambiguation suffix stripping
# Corpus lines may carry tags like N;@2 (2nd N sibling in same path)
print(strip_disambig("N;@2"))        # N
print(strip_disambig("C-NP;@5"))     # C-NP
print(strip_disambig("LOG;@2"))      # LOG
print(strip_disambig("IP-MAT"))      # IP-MAT  (no suffix, unchanged)

# is_phon_tag handles the suffix
print(is_phon_tag("LOG;@2"))         # True
print(is_phon_tag("NP;@2"))          # False


N
C-NP
LOG
IP-MAT
True
False


In [34]:
print("Part-of-speech values (sample):")
for pos, desc in list(POS_VALUES.items())[:10]:
    print(f"  {pos:25s}  {desc}")


Part-of-speech values (sample):
  noun                       common noun
  noun (deverbal)            deverbal noun
  noun (covert)              covert/empty noun
  noun (exposed)             exposed noun head
  noun (deictic)             deictic noun
  personal name              personal name
  place name                 place name
  verb                       verb
  adjective                  adjective
  adverb                     adverb


In [35]:
print("Inflection type codes:")
for itype, desc in ITYPE_VALUES.items():
    print(f"  {itype:25s}  {desc}")


Inflection type codes:
  quadrigrade                yodan (four-grade) conjugation
  lower_bigrade              shimo-nidan (lower two-grade) conjugation
  upper_bigrade              kami-nidan (upper two-grade) conjugation
  upper_monograde            kami-ichidan (upper one-grade) conjugation
  KU-adjective               ku-katsuyō adjective
  SIKU-adjective             shiku-katsuyō adjective
  R-irregular                ra-hen irregular (ari, wori, haberi, …)
  N-irregular                na-hen irregular
  K-irregular                ka-hen irregular (ku, ko, ki)
  S-irregular                sa-hen irregular (su, se, si)
  irregular                  irregular (other)
  non-inflecting             uninflected (particles, some nouns)


In [36]:
print("VAX (verbal auxiliary) semantics:")
for sem, desc in VAX_SEMANTICS.items():
    print(f"  {sem:8s}  {desc}")


VAX (verbal auxiliary) semantics:
  NEG       negative
  STV       stative
  PRF       perfective
  SPST      simple past
  MPST      modal past
  CJR       conjectural
  PASS      passive/potential
  CTV       causative
  RSP       respect/honorific
  SJV       subjunctive


In [37]:
print("Text collection codes:")
for code, name in TEXT_COLLECTIONS.items():
    print(f"  {code:6s}  {name}")


Text collection codes:
  BS      Buddha's Footprints Stones (21 texts)
  EN      Engi-shiki Norito
  FK      Fudoki poems (Hitachi, Harima, Hizen, Tango, Ise)
  JSHT    Jōgū Shōtoku Hōtei Setsu (4 texts)
  KH      Kaifūsō / related texts (40 texts)
  KK      Kojiki kayō — songs embedded in the Kojiki (112 poems)
  MYS     Man'yōshū — Japan's earliest poetry anthology (4,685 poems)
  NSK     Nihon shoki kayō — songs embedded in the Nihon shoki (133 poems)
  SM      Shoku-Nihongi Senmyō
  SNK     Shoku Nihon Kōki or related texts


In [38]:
print("Geographic dialect regions:")
for code, desc in GEO_VALUES.items():
    print(f"  {code:8s}  {desc}")


Geographic dialect regions:
  EOJ       Eastern Old Japanese
  NEOJ      North-Eastern Old Japanese
  SEOJ      South-Eastern Old Japanese
  CEOJ      Central-Eastern Old Japanese
  UEOJ      Upper-Eastern Old Japanese
